# House Price Prediction - Phase 2: Final System & Showcase
### Ames Housing Dataset (Kaggle House Prices)

| Step | Topic |
|------|-------|
| 1 | Feature Engineering |
| 2 | Advanced Models |
| 3 | Cross-Validation & GridSearchCV |
| 4 | Model Comparison Dashboard |
| 5 | Visualizations & Insights |
| 6 | Save Models |


## Step 1: Load Data & Feature Engineering
### 1a. Load & Clean

In [ ]:
import pandas as pd, numpy as np, os
import matplotlib.pyplot as plt, seaborn as sns
import warnings, pickle, json
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)


In [ ]:
import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/train.csv')
df.drop(columns=['Id'], inplace=True, errors='ignore')
TARGET = 'SalePrice'

NONE_COLS = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
for c in NONE_COLS:
    if c in df: df[c] = df[c].fillna('None')

df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
for c in ['GarageYrBlt','GarageArea','GarageCars','BsmtFinSF1','BsmtFinSF2',
          'BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath','MasVnrArea']:
    if c in df: df[c] = df[c].fillna(0)
for c in df.select_dtypes('object'):
    df[c] = df[c].fillna(df[c].mode()[0])

print(f"Shape: {df.shape}  |  Nulls remaining: {df.isnull().sum().sum()}")


### 1b. Feature Engineering

In [ ]:
df['HouseAge']        = df['YrSold']  - df['YearBuilt']
df['YearsSinceRemod'] = df['YrSold']  - df['YearRemodAdd']
df['TotalSF']         = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['TotalBath']       = (df['FullBath']    + 0.5*df['HalfBath'] +
                         df['BsmtFullBath']+ 0.5*df['BsmtHalfBath'])
df['TotalPorch']      = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                         df['3SsnPorch']   + df['ScreenPorch'])
df['HasPool']         = (df['PoolArea']    > 0).astype(int)
df['HasGarage']       = (df['GarageArea']  > 0).astype(int)
df['HasBsmt']         = (df['TotalBsmtSF'] > 0).astype(int)
df['HasFireplace']    = (df['Fireplaces']  > 0).astype(int)
df['IsNew']           = (df['YearBuilt']   >= 2000).astype(int)
df['LivAreaPerRoom']  = df['GrLivArea'] / (df['TotRmsAbvGrd'] + 1)

ENG = ['HouseAge','YearsSinceRemod','TotalSF','TotalBath','TotalPorch',
       'HasPool','HasGarage','HasBsmt','HasFireplace','IsNew','LivAreaPerRoom']
print(f"Created {len(ENG)} engineered features: {ENG}")
print(f"New shape: {df.shape}")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

cat_cols = df.select_dtypes('object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

TARGET = 'SalePrice'
X = df.drop(columns=[TARGET])
y = np.log1p(df[TARGET])   # log-transform target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Features: {X_train.shape[1]}  |  Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print("Log-transformed SalePrice target ready.")


## Step 2: Advanced Models

In [ ]:
from sklearn.linear_model  import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble      import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics       import mean_absolute_error, mean_squared_error, r2_score

def eval_model(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    pred_real = np.expm1(model.predict(Xte))
    true_real = np.expm1(yte)
    mae  = mean_absolute_error(true_real, pred_real)
    rmse = np.sqrt(mean_squared_error(true_real, pred_real))
    r2   = r2_score(true_real, pred_real)
    print(f"  {name:<28}  MAE ${mae:>9,.0f}  RMSE ${rmse:>9,.0f}  R2 {r2:.4f}")
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, '_m': model}

print(f"  {'Model':<28}  {'MAE':>13}  {'RMSE':>13}  {'R2':>6}")
print("  " + "-"*70)
res = []
res.append(eval_model("Linear Regression",  LinearRegression(),  X_train_sc, y_train, X_test_sc, y_test))
res.append(eval_model("Ridge Regression",   Ridge(alpha=10),     X_train_sc, y_train, X_test_sc, y_test))
res.append(eval_model("Lasso Regression",   Lasso(alpha=0.0005), X_train_sc, y_train, X_test_sc, y_test))
res.append(eval_model("ElasticNet",         ElasticNet(alpha=0.0005, l1_ratio=0.5), X_train_sc, y_train, X_test_sc, y_test))
res.append(eval_model("Random Forest",      RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1), X_train_sc, y_train, X_test_sc, y_test))
res.append(eval_model("Gradient Boosting",  GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42), X_train_sc, y_train, X_test_sc, y_test))


## Step 3: Cross-Validation & Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import cross_val_score, GridSearchCV

gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                               max_depth=4, subsample=0.8, random_state=42)
cv_r2 = cross_val_score(gb, X_train_sc, y_train, cv=5, scoring='r2', n_jobs=-1)
print(f"Gradient Boosting 5-Fold CV  R2: {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")


In [ ]:
print("GridSearchCV: Ridge...")
ridge_gs = GridSearchCV(Ridge(), {'alpha':[0.1,1,5,10,50,100,300]}, cv=5, scoring='r2', n_jobs=-1)
ridge_gs.fit(X_train_sc, y_train)
print(f"  Best alpha={ridge_gs.best_params_['alpha']}  CV R2={ridge_gs.best_score_:.4f}")
best_ridge = ridge_gs.best_estimator_


In [ ]:
print("GridSearchCV: Random Forest (this takes ~2-3 min)...")
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {'n_estimators':[200,400], 'max_depth':[None,20], 'min_samples_split':[2,5]},
    cv=3, scoring='r2', n_jobs=-1, verbose=1
)
rf_gs.fit(X_train_sc, y_train)
print(f"  Best: {rf_gs.best_params_}  CV R2={rf_gs.best_score_:.4f}")
best_rf = rf_gs.best_estimator_


## Step 4: Model Comparison Dashboard

In [ ]:
res_df = pd.DataFrame([{k:v for k,v in r.items() if k!='_m'} for r in res])
res_df = res_df.sort_values('R2', ascending=False).reset_index(drop=True)

print("\n" + "="*64)
print("      HOUSE PRICE PREDICTION - MODEL LEADERBOARD")
print("="*64)
print(res_df[['Model','MAE','RMSE','R2']].to_string(index=False))
print("="*64)
best = res_df.iloc[0]
print(f"\nBest: {best['Model']}  R2={best['R2']:.4f}  MAE=${best['MAE']:,.0f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = sns.color_palette('husl', len(res_df))
for ax, metric, title in zip(axes,
        ['MAE','RMSE','R2'],
        ['MAE (lower=better)','RMSE (lower=better)','R2 (higher=better)']):
    bars = ax.barh(res_df['Model'], res_df[metric], color=colors, edgecolor='white', alpha=0.9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=8)
plt.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig('model_comparison.png', dpi=150); plt.show()


## Step 5: Visualizations & Insights
### 5a. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=X_train.columns)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('RdYlGn', 20)[::-1]
top20.sort_values().plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.savefig('feature_importance.png', dpi=150); plt.show()
print("Top 10:")
print(top20.head(10).to_string())


### 5b. Predicted vs Actual — Gradient Boosting

In [ ]:
gb_final = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                     max_depth=4, subsample=0.8, random_state=42)
gb_final.fit(X_train_sc, y_train)
y_pred_real = np.expm1(gb_final.predict(X_test_sc))
y_true_real = np.expm1(y_test)
residuals   = y_true_real - y_pred_real

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
lim = [y_true_real.min()*0.9, y_true_real.max()*1.04]
axes[0].scatter(y_true_real, y_pred_real, alpha=0.4, color='steelblue', s=14)
axes[0].plot(lim, lim, 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel('Actual ($)'); axes[0].set_ylabel('Predicted ($)')
axes[0].set_title('Gradient Boosting: Predicted vs Actual', fontsize=12, fontweight='bold')
axes[0].legend()

axes[1].scatter(y_pred_real, residuals, alpha=0.4, color='coral', s=14)
axes[1].axhline(0, color='black', lw=1.5, ls='--')
axes[1].set_xlabel('Predicted ($)'); axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot — Gradient Boosting', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('phase2_predictions.png', dpi=150); plt.show()

mae  = mean_absolute_error(y_true_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
r2   = r2_score(y_true_real, y_pred_real)
print(f"Gradient Boosting:  MAE=${mae:,.0f}  RMSE=${rmse:,.0f}  R2={r2:.4f}")


### 5c. Error Distribution

In [ ]:
pct_err = np.abs(residuals) / y_true_real * 100
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(residuals, bins=50, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', ls='--', lw=1.5)
axes[0].set_title('Residuals Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Residual ($)'); axes[0].set_ylabel('Count')

axes[1].hist(pct_err, bins=50, color='gold', edgecolor='white', alpha=0.85)
axes[1].axvline(pct_err.median(), color='red', ls='--', lw=1.5,
                label=f'Median: {pct_err.median():.1f}%')
axes[1].set_title('Absolute % Error Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('% Error'); axes[1].legend()
plt.tight_layout(); plt.savefig('error_distribution.png', dpi=150); plt.show()

print(f"Median absolute % error : {pct_err.median():.2f}%")
print(f"90th percentile error   : {pct_err.quantile(0.90):.2f}%")
print(f"Predictions within 10%  : {(pct_err < 10).mean()*100:.1f}% of test set")


## Step 6: Save Final Models

In [ ]:
import pickle, json
os.makedirs('models', exist_ok=True)
with open('models/gradient_boosting_final.pkl','wb') as f: pickle.dump(gb_final, f)
with open('models/random_forest_final.pkl','wb')     as f: pickle.dump(best_rf, f)
with open('models/ridge_final.pkl','wb')             as f: pickle.dump(best_ridge, f)
with open('models/scaler_phase2.pkl','wb')           as f: pickle.dump(scaler, f)
with open('models/feature_names_phase2.json','w')    as f: json.dump(list(X_train.columns), f)

print("Saved:")
for fn in sorted(os.listdir('models')): print(f"  models/{fn}")
print("\nPhase 2 complete! Run: streamlit run app.py")
